# Build SCD2 history from repeated saves

**Track:** sql | **Difficulty:** medium | **Tags:** scd2, slowly changing dimension, window functions, nulls

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

Free Colab runs only two or three notebooks at a time. When you are done here, end the
session with Runtime > Disconnect and delete runtime before opening the next problem.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("004-scd2-history-from-saves")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "customer_saves": """
UEFSMRUEFYAFFdoCTBVQFQASAADAAggvAQAFAQA8DQgAOQ0IADUNCAA+DQgAUA0IAE8NCAA/DQgAQw0IAEcNCABADQgAQQ0IAFIN
CABUDQgASw0IADgNCAAxDQgARg0IAEgNCAA2DQgAMA0IADcNCABEDQgALg0IADsNCABTDQgATA0IADQNCABFDQgAPQ0IADINCABC
DQgATQ0IAEkNCABODQgAOg0IAFENCABKDQg8LQEAAAAAAAAzAQAAAAAAABUAFdIBFdgBLBX2ARUQFQYVBhwYCFQBAAAAAAAAGAgt
AQAAAAAAABYAKAhUAQAAAAAAABgILQEAAAAAAAAREQAAAGnwaAMAAAD2AQEGIUAgDERhHEiQKAvTOIuSPJBQRIo0UdMkFEQFWdU1
EFiGadhGcF0GedanaGAWYoMjVeKIjEKRBFuEAQoJMdcSkt2mlZ5gAk+gQVR1HEyUFII0Wc+zLU7GeNVJQlH2AAAAABUEFbwBFboB
TBUSFQASAABeeAQAAABQdW5lCQAAAEh5ZGVyYWJhZAYAAABNdW1iYWkBF3BCZW5nYWx1cnUHAAAAS29sa2F0YQUAAABEZWxoaQEJ
EEtvY2hpATRASmFpcHVyBwAAAENoZW5uYWkVABWiARWoASwV9gEVEBUGFQYcNgooBFB1bmUYCUJlbmdhbHVydRERAAAAUfBQDwAA
AAO/MAEF/r8aAQP+XAED/gQfEDJCBWFxcnSBhYIwU1NDFHGBA4MiUhNIgRgHNTAhMUA0FjMkVDR4UzhHIQMlB4NVUzg0g1dTVAKC
NVMAFQQVsAYVugZMFcwBFQASAACYA/SXAVZNAABLTQAAcE0AAFJNAABXTQAABE4AAHdNAABiTQAAD00AAF1NAABsTQAAjU0AAKdN
AAAMTQAAZ00AANhNAACOTQAAI00AAMBNAAAuTQAAWk0AABNOAACYTQAAJk0AAGlNAADVTQAAYE0AAH5NAAA4TQAACk4AALBNAADI
TQAAKE0AAKhNAACkTQAAMU0AAGFNAADPTQAAB04AAENNAAAxTgAAEk0AAMRNAACITQAAhU0AAF9NAABmTQAASU0AAGtNAAAUTQAA
uk0AAFVNAAALTgAA9E0AAEFNAAARTQAAXk0AAExNAAA+TQAAZE0AADlNAADdTQAAVE0AAKpNAABtTQAAkU0AAPJNAAAsTQAARk0A
ALhNAAAYTQAAPU0AAItNAABjTQAAUE0AAKtNAAAgTQAAek0AALZNAACaTQAAc00AALJNAAC3TQAAlk0AAFhNAABTTQAAcU0AAMZN
AABoTQAApk0AAPpNAACzTQAAKk0AAKBNAAA2TQAAC00AAGNOAAAXTQAAvU0AAHhNAADOTQAAGE4AABUAFfIBFfgBLBX2ARUQFQYV
BhwYBGNOAAAYBAtNAAAWACgEY04AABgEC00AABERAAAAefB4AwAAAPYBAQchgIBgQCgYDoiEQrBgNBwPSEQyoVQsF0xGs+F0PB9Q
SDQ6kEqmEyqlWrFaloXrBYvJSDPaqway3XA5gm7Hm0l6vh8wKBwSi8YjMqncLJlNJ+YJjUqn1GnVesX+stotgdu9WL1fMCUsxo6L
5GIZAAAAABUEGUw1ABgGc2NoZW1hFQYAFQQlAhgLY3VzdG9tZXJfaWQAFQwlAhgEY2l0eSUATBwAAAAVAiUCGAhzYXZlZF9vbiUM
TGwAAAAW9gEZHBk8JgAcFQQZNQAGEBkYC2N1c3RvbWVyX2lkFQIW9gEW9gcW1gUmggMmCBwYCFQBAAAAAAAAGAgtAQAAAAAAABYA
KAhUAQAAAAAAABgILQEAAAAAAAAREQAZLBUEFQAVAgAVABUQFQIAPCkGGSYA9gEAAAAmABwVDBk1AAYQGRgEY2l0eRUCFvYBFtQD
FtgDJrgHJt4FHDYKKARQdW5lGAlCZW5nYWx1cnUREQAZLBUEFQAVAgAVABUQFQIAPBbgDBkGGSYK7AEAAAAmABwVAhk1AAYQGRgI
c2F2ZWRfb24VAhb2ARaoCRa4CSaSECa2CRwYBGNOAAAYBAtNAAAWACgEY04AABgEC00AABERABksFQQVABUCABUAFRAVAgA8KQYZ
JgD2AQAAABbyFBb2ASYIFuYSABkcGAxBUlJPVzpzY2hlbWEYwAIvLy8vLytnQUFBQVFBQUFBQUFBS0FBd0FCZ0FGQUFnQUNnQUFB
QUFCQkFBTUFBQUFDQUFJQUFBQUJBQUlBQUFBQkFBQUFBTUFBQUI4QUFBQVBBQUFBQVFBQUFDZy8vLy9BQUFCQ0JBQUFBQWdBQUFB
QkFBQUFBQUFBQUFJQUFBQWMyRjJaV1JmYjI0QUFBWUFDQUFHQUFZQUFBQUFBQUFBMVAvLy93QUFBUVVRQUFBQUhBQUFBQVFBQUFB
QUFBQUFCQUFBQUdOcGRIa0FBQUFBQkFBRUFBUUFBQUFRQUJRQUNBQUdBQWNBREFBQUFCQUFFQUFBQUFBQUFRSVFBQUFBSkFBQUFB
UUFBQUFBQUFBQUN3QUFBR04xYzNSdmJXVnlYMmxrQUFnQURBQUlBQWNBQ0FBQUFBQUFBQUZBQUFBQQAYIHBhcnF1ZXQtY3BwLWFy
cm93IHZlcnNpb24gMjUuMC4xGTwcAAAcAAAcAAAABQMAAFBBUjE=
""",
}

_data_dir = pathlib.Path("data") / "004-scd2-history-from-saves"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

customer_saves = spark.read.parquet((_data_dir / "customer_saves.parquet").resolve().as_posix())
customer_saves.createOrReplaceTempView("customer_saves")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        # Returns nothing, so a notebook cell ending in check(result) prints only the message.

    return check

EXPECTED = {'column_fingerprints': {'city': '05dfd67ef8fcc8b9de1b094e0b37f26b79c7361a70ea32ac8726c61551be28be',
                         'customer_id': 'cbdb11f52b5145bd04762927150e988813919c0380f255b91771e123299e4c5f',
                         'is_current': 'a8468634cfebe14711152ceb101f36cff4cded38f28b7d33245a663beef4804f',
                         'valid_from': 'f9abe5ee01fd3b2087c0fa56ee039b0fea4b3d339676bf9d997f8fba62dfe1a6',
                         'valid_to': 'd682a480234d1fc2957b7d26c60776e7af512e04a297e90f28551b25ff9c9b95'},
 'columns': [['city', 'string'],
             ['customer_id', 'bigint'],
             ['is_current', 'boolean'],
             ['valid_from', 'date'],
             ['valid_to', 'date']],
 'fingerprint': '5547490eaec502cf6eae149659beb9c249bb4b1a69b4ab53d45c184b7d311756',
 'order_matters': False,
 'row_count': 85}

check = make_check(EXPECTED)

## The situation

The CRM writes a row to `customer_saves` every time someone presses Save on a customer
record, whether or not anything changed. Analysts need the history of where each
customer lived as a type 2 slowly changing dimension: one row per period during which
the city stayed the same.

## Input tables

### `customer_saves`

One row per save. A customer has at most one save per date.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | The customer. |
| `city` | STRING | The city on the record after the save. NULL means no city on record. |
| `saved_on` | DATE | The date of the save. |

## Requirement

Return one row per version of a customer's city.

- A new version starts at the customer's first save, and at every save where the
  city is different from the city of the previous save.
- A save that leaves the city unchanged does not start a new version.
- NULL is a value like any other here. Going from a city to NULL, or from NULL to a
  city, is a change. Two consecutive saves with a NULL city are not.
- If a customer returns to a city they lived in before, that is a new version. The
  two stays are separate rows.
- `valid_from` is the `saved_on` of the save that started the version.
- `valid_to` is the `valid_from` of the customer's next version. The latest version
  has `valid_to` = NULL.
- `is_current` is true for the latest version of each customer and false otherwise.

## Expected output

One row per customer version.

| Column | Type | Description |
|---|---|---|
| `customer_id` | BIGINT | |
| `city` | STRING | Can be NULL. |
| `valid_from` | DATE | |
| `valid_to` | DATE | NULL for the current version. |
| `is_current` | BOOLEAN | |

In [ ]:
customer_saves.show(5)

## Your answer

Write one Spark SQL query. The tables are available as views: `customer_saves`.

In [ ]:
query = """
-- Write your Spark SQL query here.
"""
result = spark.sql(query)
result.show()

In [ ]:
check(result)